# Azure Data Engineering — First Contact

Azure data engineering fits a simple flow: land data in a durable lake, analyze it with SQL or Spark, orchestrate movement with pipelines, and handle streaming with an event backbone. In Azure, that mental model is usually **ADLS Gen2 → Synapse → Data Factory → Event Hubs**. Think of ADLS Gen2 as the storage foundation, Synapse as the analytics workspace, Data Factory as the orchestration layer, and Event Hubs as the real-time ingestion path.

ADLS Gen2 is Azure Blob Storage with a **hierarchical namespace** turned on, which gives you folder-like behavior, path-based operations, and POSIX-style access control patterns. That makes it a better fit for analytics than flat object storage alone. Synapse is Azure’s unified analytics surface: SQL querying, Spark processing, and pipeline integration live together in one workspace instead of being scattered across multiple tools.

Many financial institutions are Azure-first due to Microsoft enterprise agreements. Citi uses Azure Synapse for SQL Analytics and ADLS Gen2 as the data lake foundation. This notebook uses a Citi-style telemetry example to show how Azure pieces fit together for batch and streaming data engineering.

```text
[Postgres] → [ADLS Gen2] → [Synapse SQL] ← [Event Hubs → Synapse Streaming]
```


In [1]:
# Packages pre-installed: azure-storage-file-datalake azure-eventhub azure-identity psycopg2-binary pandas pyarrow
from azure.storage.filedatalake import DataLakeServiceClient
from azure.eventhub import EventData, EventHubConsumerClient, EventHubProducerClient
from azure.identity import DefaultAzureCredential
import psycopg2
import pandas as pd
import io
import json
import os
import subprocess
import random
import string
import threading
import time

SUBSCRIPTION_ID = "b3811436-61fc-4a3a-a6a9-deb05955076d"

SUFFIX = ''.join(random.choices(string.ascii_lowercase + string.digits, k=6))
RG_NAME        = "citi-telemetry-rg"
STORAGE_NAME   = f"cititelemetry{SUFFIX}"[:24].lower().replace("-", "")
LOCATION       = "eastus"
EVENTHUB_NAMESPACE = f"citi-events-{SUFFIX}"
SYNAPSE_WORKSPACE  = f"citi-synapse-{SUFFIX}"
CONTAINER_NAME = "telemetry"

print(f"Subscription : {SUBSCRIPTION_ID}")
print(f"Suffix       : {SUFFIX}")
print(f"Storage Acct : {STORAGE_NAME}")
print(f"EventHub NS  : {EVENTHUB_NAMESPACE}")

Subscription : b3811436-61fc-4a3a-a6a9-deb05955076d
Suffix       : welcqx
Storage Acct : cititelemetrywelcqx
EventHub NS  : citi-events-welcqx


Create Azure resources via CLI (subprocess) and Python SDK

In [2]:
AZ = r"C:\Program Files (x86)\Microsoft SDKs\Azure\CLI2\wbin\az.cmd"

def run_az(cmd):
    full_cmd = [AZ] + cmd
    result = subprocess.run(full_cmd, capture_output=True, text=True, encoding="utf-8", errors="replace")
    if result.returncode != 0:
        raise RuntimeError(
            f"Azure CLI command failed:\nCOMMAND: az {' '.join(cmd)}\nSTDOUT: {result.stdout}\nSTDERR: {result.stderr}"
        )
    return result

# Pin subscription and register providers (required for new free accounts)
run_az(["account", "set", "--subscription", SUBSCRIPTION_ID])
print(f"Subscription set: {SUBSCRIPTION_ID}")

for provider in ["Microsoft.Storage", "Microsoft.EventHub"]:
    run_az(["provider", "register", "--namespace", provider])
    print(f"Provider registered: {provider}")

# Create resource group
run_az(["group", "create", "--name", RG_NAME, "--location", LOCATION,
        "--subscription", SUBSCRIPTION_ID])
print(f"Resource group '{RG_NAME}' ready")

# Create ADLS Gen2 storage account
run_az([
    "storage", "account", "create",
    "--name", STORAGE_NAME,
    "--resource-group", RG_NAME,
    "--location", LOCATION,
    "--sku", "Standard_LRS",
    "--kind", "StorageV2",
    "--hns", "true",
    "--subscription", SUBSCRIPTION_ID
])

storage_keys_result = run_az([
    "storage", "account", "keys", "list",
    "--resource-group", RG_NAME,
    "--account-name", STORAGE_NAME,
    "--query", "[0].value",
    "-o", "tsv",
    "--subscription", SUBSCRIPTION_ID
])
STORAGE_KEY = storage_keys_result.stdout.strip()
if not STORAGE_KEY:
    raise ValueError("Failed to retrieve storage account key.")

run_az([
    "storage", "container", "create",
    "--name", CONTAINER_NAME,
    "--account-name", STORAGE_NAME,
    "--account-key", STORAGE_KEY
])

print(f"ADLS Gen2 account '{STORAGE_NAME}' ready with container '{CONTAINER_NAME}'")

Subscription set: b3811436-61fc-4a3a-a6a9-deb05955076d


Provider registered: Microsoft.Storage


Provider registered: Microsoft.EventHub


Resource group 'citi-telemetry-rg' ready


ADLS Gen2 account 'cititelemetrywelcqx' ready with container 'telemetry'


## ADLS Gen2 — Azure Data Lake Storage

ADLS Gen2 is Azure’s analytics-focused storage layer. The **hierarchical namespace** allows directory-level operations such as rename and atomic move, which matters for data engineering workflows that promote files from raw to curated zones. It also supports POSIX-style ACL behavior, making lake permissions more structured than plain blob access.

For data engineers, Parquet on ADLS Gen2 is Azure’s close equivalent to Parquet on S3. The pattern is the same: land structured files into the lake, partition or organize them by folder, and query them later from analytics engines such as Synapse Serverless SQL or Spark.


In [3]:
pg_conn = psycopg2.connect(
    host="localhost",
    port=5432,
    dbname="de_telemetry",
    user="de_admin",
    password="DeAdmin2026!"
)

endpoints_df = pd.read_sql_query(
    "SELECT endpoint_id, name, region, status, category FROM endpoints ORDER BY endpoint_id",
    pg_conn
)
alerts_df = pd.read_sql_query(
    "SELECT alert_id, endpoint_id, severity, message, created_at FROM alerts ORDER BY alert_id",
    pg_conn
)

pg_conn.close()

endpoints_buffer = io.BytesIO()
alerts_buffer = io.BytesIO()

endpoints_df.to_parquet(endpoints_buffer, index=False)
alerts_df.to_parquet(alerts_buffer, index=False)

endpoints_buffer.seek(0)
alerts_buffer.seek(0)

account_url = f"https://{STORAGE_NAME}.dfs.core.windows.net"
service_client = DataLakeServiceClient(account_url=account_url, credential=STORAGE_KEY)
file_system_client = service_client.get_file_system_client(CONTAINER_NAME)

for directory in ["endpoints", "alerts"]:
    try:
        file_system_client.create_directory(directory)
    except Exception:
        pass

endpoints_file_client = file_system_client.get_file_client("endpoints/endpoints.parquet")
alerts_file_client = file_system_client.get_file_client("alerts/alerts.parquet")

endpoints_file_client.upload_data(endpoints_buffer.getvalue(), overwrite=True)
alerts_file_client.upload_data(alerts_buffer.getvalue(), overwrite=True)

print(f"Uploaded {len(endpoints_df)} endpoints and {len(alerts_df)} alerts to ADLS Gen2")


C:\Users\shareuser\AppData\Local\Temp\ipykernel_70344\4027313965.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  endpoints_df = pd.read_sql_query(
C:\Users\shareuser\AppData\Local\Temp\ipykernel_70344\4027313965.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  alerts_df = pd.read_sql_query(


Uploaded 10000 endpoints and 25000 alerts to ADLS Gen2


## Azure Synapse — Serverless SQL

Synapse Serverless SQL Pool can query files in ADLS Gen2 directly by using `OPENROWSET`, which is conceptually similar to Athena querying data in S3. There is no ingestion step required for simple exploration, and pricing is based on data scanned rather than provisioned cluster uptime. It works especially well with Parquet and Delta formats.

Full Synapse workspace setup is easiest in the Azure portal. Once the workspace exists, the following SQL can be run inside Synapse Studio against the lake files uploaded in this notebook.

```sql
-- Run in Synapse Studio

-- Create external data source pointing to ADLS Gen2
CREATE EXTERNAL DATA SOURCE CityTelemetry
WITH (
    LOCATION = 'https://{STORAGE_NAME}.dfs.core.windows.net/telemetry'
);

-- Query Parquet files directly
SELECT severity, COUNT(*) as cnt
FROM OPENROWSET(
    BULK 'alerts/alerts.parquet',
    DATA_SOURCE = 'CityTelemetry',
    FORMAT = 'PARQUET'
) AS alerts
GROUP BY severity
ORDER BY cnt DESC;
```

Create a free Synapse workspace at portal.azure.com — takes 5 minutes.


## Azure Event Hubs — Kafka-Compatible Streaming

Event Hubs is Azure’s large-scale event ingestion service. Conceptually, it plays the role that Kafka or Kinesis often plays in other clouds: partitioned event streams, consumer groups, replay within retention windows, and downstream analytics integration. Azure also supports Kafka protocol compatibility, so Kafka clients can often target Event Hubs with configuration changes instead of a full rewrite.

For telemetry pipelines, Event Hubs commonly sits in front of stream processors, real-time dashboards, and anomaly detection jobs. In enterprise environments, it is a standard way to ingest operational events before landing them into analytics platforms like Synapse.


In [4]:
# Standard tier supports 7-day default retention (Basic only allows 1 day, conflicts with CLI default)
run_az([
    "eventhubs", "namespace", "create",
    "--name", EVENTHUB_NAMESPACE,
    "--resource-group", RG_NAME,
    "--location", LOCATION,
    "--sku", "Standard",
    "--subscription", SUBSCRIPTION_ID
])

run_az([
    "eventhubs", "eventhub", "create",
    "--name", "citi-alerts",
    "--namespace-name", EVENTHUB_NAMESPACE,
    "--resource-group", RG_NAME,
    "--partition-count", "2"
])

eh_conn_result = run_az([
    "eventhubs", "namespace", "authorization-rule", "keys", "list",
    "--resource-group", RG_NAME,
    "--namespace-name", EVENTHUB_NAMESPACE,
    "--name", "RootManageSharedAccessKey",
    "--query", "primaryConnectionString",
    "-o", "tsv"
])
EVENTHUB_CONNECTION_STR = eh_conn_result.stdout.strip()
if not EVENTHUB_CONNECTION_STR:
    raise ValueError("Failed to retrieve Event Hubs connection string.")

producer = EventHubProducerClient.from_connection_string(
    conn_str=EVENTHUB_CONNECTION_STR,
    eventhub_name="citi-alerts"
)

sample_alerts = []
for _, row in alerts_df.head(10).iterrows():
    sample_alerts.append({
        "alert_id": int(row["alert_id"]),
        "endpoint_id": int(row["endpoint_id"]),
        "severity": str(row["severity"]),
        "message": str(row["message"]),
        "created_at": str(row["created_at"])
    })

event_batch = producer.create_batch()
for alert in sample_alerts:
    payload = json.dumps(alert).encode("utf-8")
    event_batch.add(EventData(payload))

producer.send_batch(event_batch)
producer.close()

print("Sent 10 events to Event Hubs citi-alerts")

Sent 10 events to Event Hubs citi-alerts


In [5]:
received_events = []

def on_event(partition_context, event):
    payload = event.body_as_str(encoding="UTF-8")
    received_events.append(payload)
    print(payload)
    partition_context.update_checkpoint(event)

consumer = EventHubConsumerClient.from_connection_string(
    conn_str=EVENTHUB_CONNECTION_STR,
    consumer_group="$Default",
    eventhub_name="citi-alerts"
)

stop_timer = threading.Timer(8.0, consumer.close)
stop_timer.start()

try:
    consumer.receive(
        on_event=on_event,
        starting_position="-1",
        max_wait_time=5
    )
except Exception as exc:
    if "handler has been shut down" not in str(exc).lower():
        raise
finally:
    stop_timer.cancel()
    try:
        consumer.close()
    except Exception:
        pass

count = len(received_events)
print(f"Received {count} events from Event Hubs")


{"alert_id": 1, "endpoint_id": 7416, "severity": "CRITICAL", "message": "Health check failed: HTTP 503 from email-37.stevens.com", "created_at": "2026-01-13 11:40:27.063609+00:00"}


{"alert_id": 2, "endpoint_id": 3694, "severity": "CRITICAL", "message": "TCP connection pool exhausted on port 5432", "created_at": "2026-01-24 11:42:23.063609+00:00"}
{"alert_id": 3, "endpoint_id": 5828, "severity": "HIGH", "message": "Memory usage at 95% \u2014 potential OOM imminent", "created_at": "2026-01-16 11:11:06.063609+00:00"}


{"alert_id": 4, "endpoint_id": 1986, "severity": "MEDIUM", "message": "Network throughput dropped below SLA threshold", "created_at": "2026-01-01 06:01:32.063609+00:00"}
{"alert_id": 5, "endpoint_id": 4807, "severity": "CRITICAL", "message": "SSL certificate expires in 7 days", "created_at": "2026-01-12 16:04:36.063609+00:00"}


{"alert_id": 6, "endpoint_id": 907, "severity": "HIGH", "message": "Network throughput dropped below SLA threshold", "created_at": "2026-01-26 13:42:25.063609+00:00"}
{"alert_id": 7, "endpoint_id": 5549, "severity": "CRITICAL", "message": "SSL certificate expires in 7 days", "created_at": "2026-02-14 12:18:48.063609+00:00"}


{"alert_id": 8, "endpoint_id": 8778, "severity": "LOW", "message": "Health check failed: HTTP 503 from db-33.hansen-jackson.com", "created_at": "2025-12-31 17:27:54.063609+00:00"}


{"alert_id": 9, "endpoint_id": 8911, "severity": "HIGH", "message": "Process maryjackson exited unexpectedly \u2014 restart triggered", "created_at": "2026-02-05 14:23:30.063609+00:00"}
{"alert_id": 10, "endpoint_id": 3987, "severity": "MEDIUM", "message": "Network throughput dropped below SLA threshold", "created_at": "2026-02-22 14:52:30.063609+00:00"}


EventProcessor instance 'fdb953d8-b051-4781-b328-f286a92428ee' of eventhub 'citi-alerts' partition '1' consumer group '$Default'. An error occurred while receiving. The exception is AttributeError("'NoneType' object has no attribute 'body_as_str'").


Received 10 events from Event Hubs


## Azure vs AWS vs GCP — Service Mapping

| Function | AWS | Azure | GCP |
|----------|-----|-------|-----|
| Object storage | S3 | Blob Storage | Cloud Storage |
| Data lake | S3 + Lake Formation | ADLS Gen2 | GCS + Data Catalog |
| Serverless SQL | Athena | Synapse Serverless | BigQuery |
| Managed Spark | EMR | Synapse Spark / Databricks | Dataproc |
| ETL/pipeline | Glue ETL | Azure Data Factory | Cloud Dataflow |
| Event streaming | Kinesis | Event Hubs | Pub/Sub |
| Data warehouse | Redshift | Synapse Dedicated Pool | BigQuery |
| Governance | Lake Formation | Microsoft Purview | Dataplex |


In [6]:
run_az([
    "group", "delete",
    "--name", RG_NAME,
    "--yes",
    "--no-wait"
])

print(f"Resource group '{RG_NAME}' deletion initiated (runs in background)")

Resource group 'citi-telemetry-rg' deletion initiated (runs in background)


## What Just Happened

- Created an Azure resource group and an ADLS Gen2 storage account.
- Pulled telemetry data from local Postgres and uploaded Parquet files into ADLS Gen2.
- Showed the Synapse Serverless SQL pattern for querying lake files directly.
- Created an Event Hubs namespace and event hub.
- Published 10 telemetry alert events and consumed them back from Event Hubs.
- Triggered cleanup by deleting the resource group.

Azure is Citi's primary cloud in EMEA. ADLS Gen2 + Synapse + ADF is the standard Citi data platform — the same pattern as AWS but with Microsoft tooling.

Next: Run `multicloud_concepts.md` for service mapping and when to choose which cloud.
